# Build `master_dataset.csv`
State × fiscal-year panel combining RBI BSR-1 district credit (r1222) with state income, CPI and national series.

**Before running:** put the 9 csv files in one folder and set `DATA_DIR` in the code cell. To test on `sample.csv` first, set `SAMPLE_MODE = True`.

The cell prints a coverage table at the end. Check that `nsdp_manuf_cnst` and `deflator` cover about 30 states from 2010, and that no `!! unmapped state codes` warning appears.

In [2]:
# =====================================================================
# Build master_dataset.csv : state x fiscal-year panel
# Bank credit (RBI BSR-1, r1222) + state income + CPI + national series
# =====================================================================
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------- 1. SETTINGS (edit these) --------------------------
DATA_DIR = Path(".")                 # folder that holds the 9 csv files
SAMPLE_MODE = False                  # True -> read the single sample.csv instead
SAMPLE_PATH = DATA_DIR / "sample.csv"
SAMPLE_SCALE = 10                    # sample is ~10% of rows -> scale credit sums x10
YEAR_MIN, YEAR_MAX = 2010, 2026      # fiscal-year-end range kept in the panel
SMALL_TICKET_LAKH = 25               # avg loan per account (Rs lakh) that counts as "small ticket"
OUT_PATH = DATA_DIR / "master_dataset.csv"

FILES = {
    "credit": "r1222_bank_credit_of_scbs_bank_group_population_group_occupatio.csv",
    "nsdp":   "comp_eco_act_nsdp_fact_cst_rn.csv",
    "gsdp":   "gross_state_dom_prd_rn.csv",
    "pcnsdp": "per_cap_net_sdp_rn.csv",
    "cpi":    "cpi_ruc_st_rn.csv",
    "lna":    "lna_scb_sr_occ_bsr1_a_rn.csv",
    "asi":    "ann_surv_inds_rn.csv",
    "rrb":    "r331_important_banking_indicators_regional_rural_banks_outstand.csv",
    "shg":    "r338_self_help_group_bank_linkage_programme.csv",
}
CRORE = 1e7

# ---------------- 2. LOAD ------------------------------------------
def load_raw():
    if SAMPLE_MODE:
        s = pd.read_csv(SAMPLE_PATH, low_memory=False)
        return {k: s[s["source_file"] == f].dropna(axis=1, how="all").copy()
                for k, f in FILES.items()}
    raw = {}
    for k, f in FILES.items():
        p = DATA_DIR / f
        if k == "credit":   # large file: keep only needed columns
            need = {"row_no", "c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8", "c9", "c11", "c12"}
            raw[k] = pd.read_csv(p, low_memory=False, usecols=lambda c: c in need, dtype=str)
        else:
            raw[k] = pd.read_csv(p, low_memory=False)
        print(f"read {f:75s} {raw[k].shape}")
    return raw

def num(s):
    """'1,26,442' / '-' / ' 3.4 ' -> float"""
    return pd.to_numeric(s.astype(str).str.replace(",", "").str.strip()
                         .replace({"-": np.nan, "": np.nan, "nan": np.nan}), errors="coerce")

# ---------------- 3. STATE NAME / CODE HARMONISATION ----------------
NAME_MAP = {  # r1222 spellings -> canonical
    "DELHI": "Delhi", "NCT OF DELHI": "Delhi",
    "ANDAMAN & NICOBAR": "Andaman & Nicobar Islands", "ANDAMAN & NICOBAR ISLANDS": "Andaman & Nicobar Islands",
    "DADRA & NAGAR HAVELI": "Dadra & Nagar Haveli and Daman & Diu",
    "DAMAN & DIU": "Dadra & Nagar Haveli and Daman & Diu",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU": "Dadra & Nagar Haveli and Daman & Diu",
    "JAMMU & KASHMIR": "Jammu & Kashmir",
}
CODE_MAP = {  # RBI DBIE state codes -> canonical
    "ANN": "Andaman & Nicobar Islands", "ANP": "Andhra Pradesh", "ARP": "Arunachal Pradesh",
    "ASM": "Assam", "BIH": "Bihar", "CHD": "Chandigarh", "CHHT": "Chhattisgarh",
    "DEL": "Delhi", "NTCDEL": "Delhi", "GOA": "Goa", "GUJ": "Gujarat", "HAR": "Haryana",
    "HP": "Himachal Pradesh", "JHK": "Jharkhand", "JNK": "Jammu & Kashmir", "KAR": "Karnataka",
    "KER": "Kerala", "MAH": "Maharashtra", "MANI": "Manipur", "MEG": "Meghalaya", "MIZO": "Mizoram",
    "MP": "Madhya Pradesh", "NAGA": "Nagaland", "ORI": "Odisha", "OD": "Odisha", "PUN": "Punjab",
    "PY": "Puducherry", "RAJA": "Rajasthan", "SIKK": "Sikkim", "TEL": "Telangana", "TN": "Tamil Nadu",
    "TRI": "Tripura", "UK": "Uttarakhand", "UP": "Uttar Pradesh", "WB": "West Bengal",
    "LAK": "Lakshadweep", "DNHA": "Dadra & Nagar Haveli and Daman & Diu",
    "DADI": "Dadra & Nagar Haveli and Daman & Diu", "DNHDD": "Dadra & Nagar Haveli and Daman & Diu",
}
NON_STATE = {"ALL_INDIA", "ALL INDIA", "INDIA"}

def canon_name(s):
    s = s.astype(str).str.strip().str.upper()
    return s.map(NAME_MAP).fillna(s.str.title().str.replace(" And ", " and "))

def canon_code(s):
    s = s.astype(str).str.strip().str.upper()
    out = s.map(CODE_MAP)
    bad = sorted(set(s[out.isna() & ~s.isin(NON_STATE)]))
    if bad:
        print("  !! unmapped state codes kept as-is (add them to CODE_MAP):", bad)
    return out.fillna(s).where(~s.isin(NON_STATE))

# ---------------- 4. BANK CREDIT (r1222) ----------------------------
def build_credit(c):
    c = c.copy()
    c["year"] = num(c["c1"]).astype("Int64")
    c = c[c["year"].between(YEAR_MIN, YEAR_MAX)].copy()
    c["region"] = c["c2"].str.strip().str.title()
    c["state"] = canon_name(c["c3"])
    c["district"] = c["c4"].str.strip().str.upper()
    c["pop_group"] = c["c5"].str.upper().str.replace("-", " ").str.strip()
    c["bank_group"] = (c["c6"].str.upper().str.strip()
                       .replace({"SMALL FINANCE BANK": "SMALL FINANCE BANKS",
                                 "INDIVIDUAL REGIONAL RURAL BANKS": "REGIONAL RURAL BANKS"}))
    c["sector"] = c["c7"].str.extract(r"^\s*([IVX]+)\.", expand=False)
    c["subno"] = num(c["c8"].str.extract(r"^\s*(\d+)\.", expand=False))
    c["acc"] = num(c["c9"])
    c["lim"] = num(c["c11"])        # credit limit, Rs crore
    c["out"] = num(c["c12"])        # amount outstanding, Rs crore
    if SAMPLE_MODE:
        c[["acc", "lim", "out"]] *= SAMPLE_SCALE

    ind = c["sector"] == "II"
    c["is_manuf"] = ind & c["subno"].between(2, 16)          # manufacturing sub-sectors 2-16
    c["grassroots"] = (c["bank_group"].isin(["REGIONAL RURAL BANKS", "SMALL FINANCE BANKS", "LOCAL AREA BANKS"])
                       | c["pop_group"].isin(["RURAL", "SEMI URBAN"]))
    c["rrb_sfb"] = c["bank_group"].isin(["REGIONAL RURAL BANKS", "SMALL FINANCE BANKS", "LOCAL AREA BANKS"])
    c["rural_su"] = c["pop_group"].isin(["RURAL", "SEMI URBAN"])

    # small-ticket cells: average outstanding per account <= SMALL_TICKET_LAKH
    # (cell = district x population group x bank group x manufacturing sub-sector)
    ticket_lakh = c["out"] * 100 / c["acc"].where(c["acc"] > 0)
    c["small_ticket"] = c["is_manuf"] & (ticket_lakh <= SMALL_TICKET_LAKH)

    def w(mask, col="out"):
        return c[col].where(mask, 0)

    sec = {"I": "agri", "II": "industry", "III": "transport", "IV": "services",
           "V": "personal", "VI": "trade", "VII": "finance", "VIII": "others"}
    agg = {"cr_out_total": c["out"], "cr_lim_total": c["lim"], "cr_acc_total": c["acc"]}
    for k, v in sec.items():
        agg[f"cr_out_{v}"] = w(c["sector"] == k)
    agg.update({
        "cr_out_manuf": w(c["is_manuf"]), "cr_lim_manuf": w(c["is_manuf"], "lim"),
        "cr_acc_manuf": w(c["is_manuf"], "acc"),
        "cr_out_mining": w(ind & (c["subno"] == 1)), "cr_out_utilities": w(ind & (c["subno"] == 17)),
        "cr_out_construction": w(ind & (c["subno"] == 18)),
        "_grass_total": w(c["grassroots"]), "_grass_manuf": w(c["grassroots"] & c["is_manuf"]),
        "_rrbsfb_manuf": w(c["rrb_sfb"] & c["is_manuf"]), "_rural_manuf": w(c["rural_su"] & c["is_manuf"]),
        "_small_manuf": w(c["small_ticket"] & c["is_manuf"]),
    })
    g = pd.DataFrame(agg).join(c[["state", "year"]]).groupby(["state", "year"]).sum()
    g["n_districts"] = c.groupby(["state", "year"])["district"].nunique()
    g["region"] = c.groupby("state")["region"].agg(lambda x: x.mode().iat[0]).reindex(g.index.get_level_values(0)).values

    sd = lambda a, b: (a / b.replace(0, np.nan))
    g["avg_ticket_manuf_lakh"] = sd(g["cr_out_manuf"] * 100, g["cr_acc_manuf"])
    g["avg_ticket_total_lakh"] = sd(g["cr_out_total"] * 100, g["cr_acc_total"])
    g["small_ticket_share_manuf"] = sd(g.pop("_small_manuf"), g["cr_out_manuf"])
    g["grassroots_share_total"] = sd(g.pop("_grass_total"), g["cr_out_total"])
    g["grassroots_share_manuf"] = sd(g.pop("_grass_manuf"), g["cr_out_manuf"])
    g["rrb_sfb_share_manuf"] = sd(g.pop("_rrbsfb_manuf"), g["cr_out_manuf"])
    g["rural_semiurban_share_manuf"] = sd(g.pop("_rural_manuf"), g["cr_out_manuf"])
    g["util_manuf"] = sd(g["cr_out_manuf"], g["cr_lim_manuf"])
    g["util_total"] = sd(g["cr_out_total"], g["cr_lim_total"])
    print(f"  credit: {len(c):,} rows -> {len(g)} state-years, {g.index.get_level_values(0).nunique()} states")
    return g.reset_index()

# ---------------- 5. MACRO HELPERS (SDMX-style files) ---------------
def tidy_sdmx(d):
    d = d.copy()
    d = d[~d["state_code"].astype(str).str.upper().isin(NON_STATE)]
    d["state"] = canon_code(d["state_code"])
    d["year"] = d["repyearend"].astype(int)
    mult = 10.0 ** d.get("unit_mult", pd.Series(0, index=d.index)).fillna(0)
    d["value"] = num(d["obs_value"]) * mult
    d["base_start"] = d["base_per"].astype(str).str.extract(r"(\d{4})", expand=False).astype(float)
    return d

def splice(d, label):
    """Chain-link older base series to the newest base at the earliest overlap year.
    d: columns state, year, base_start, value -> wide series (state, year, label)."""
    out, unlinked = [], []
    for st, x in d.groupby("state"):
        bases = sorted(x["base_start"].dropna().unique(), reverse=True)
        p = x.pivot_table(index="year", columns="base_start", values="value", aggfunc="mean")
        s = p[bases[0]].dropna()
        for b in bases[1:]:
            o = p[b].dropna()
            ov = s.index.intersection(o.index)
            if len(ov) == 0:
                unlinked.append(st)           # no overlap -> older base dropped
                continue
            y0 = ov.min()
            linked = o * s[y0] / o[y0]
            # only extend BACKWARDS; never fill gaps inside the newer series with old-base values
            s = pd.concat([s, linked[linked.index < s.index.min()]]).sort_index()
        out.append(pd.DataFrame({"state": st, "year": s.index, label: s.values}))
    if unlinked:
        print(f"  {label}: older base not linked (no overlap year) for {len(set(unlinked))} states")
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["state", "year", label])

def build_macro(raw):
    pieces = []
    # --- NSDP by industry (factor cost / basic prices) ---------------
    n = tidy_sdmx(raw["nsdp"])
    n["value"] /= CRORE
    series = {  # label: (list of class codes across bases, price type)
        "nsdp_manuf_curr": (["CLIN_MANF", "CLIN_NSVA_MANF"], "CURR_PRC"),
        "nsdp_manuf_cnst": (["CLIN_MANF", "CLIN_NSVA_MANF"], "CNST_PRC"),
        "nsdp_manuf_reg_cnst": (["CLIN_MANF_REG"], "CNST_PRC"),
        "nsdp_manuf_unreg_cnst": (["CLIN_MANF_UNREG"], "CNST_PRC"),
        "nsdp_manuf_unreg_curr": (["CLIN_MANF_UNREG"], "CURR_PRC"),
        "nsva_total_curr": (["CLIN_TTL_NSVA_BAS_PRC"], "CURR_PRC"),
        "nsva_total_cnst": (["CLIN_TTL_NSVA_BAS_PRC"], "CNST_PRC"),
    }
    for lab, (codes, prc) in series.items():
        pieces.append(splice(n[n["class_inds_rn"].isin(codes) & (n["prc_typ_rn"] == prc)], lab))
    # --- GSDP ---------------------------------------------------------
    gs = tidy_sdmx(raw["gsdp"]); gs["value"] /= CRORE
    for prc, lab in [("CURR_PRC", "gsdp_curr"), ("CNST_PRC", "gsdp_cnst")]:
        pieces.append(splice(gs[gs["prc_typ_rn"] == prc], lab))
    # --- Per-capita NSDP (Rs) and implicit deflator --------------------
    pc = tidy_sdmx(raw["pcnsdp"])
    for prc, lab in [("CURR_PRC", "pc_nsdp_curr"), ("CNST_PRC", "pc_nsdp_cnst")]:
        pieces.append(splice(pc[pc["prc_typ_rn"] == prc], lab))
    # --- CPI: combined (rural+urban) general index, fiscal-year average --
    cp = tidy_sdmx(raw["cpi"])
    cp = cp[(cp["comd_item"] == "C_GIAG") & (cp["coverage_geo_rn"] == "ALL_INDIA")]
    cp = cp.groupby(["state", "year", "base_start", "time_period"], as_index=False)["value"].mean()
    fy = (cp.groupby(["state", "year", "base_start"])["value"].agg(["mean", "count"]).reset_index())
    fy = fy[fy["count"] == 12].rename(columns={"mean": "value"})        # full fiscal years only
    cpi = splice(fy, "cpi_fy")
    # rebase so FY2011-12 (year 2012) = 100 where available
    base = cpi[cpi["year"] == 2012].set_index("state")["cpi_fy"]
    cpi["cpi_fy"] = cpi["cpi_fy"] / cpi["state"].map(base) * 100
    pieces.append(cpi)
    # --- BSR-1 occupation table (cross-check; item 1 = agriculture) ----
    l = tidy_sdmx(raw["lna"].assign(base_per="NA"))
    l["value"] /= CRORE
    l["lab"] = ("lna_occ" + l["occp_gp_item"].astype(float).astype(int).astype(str) + "_"
                + l["measure_type"].map({"AMT_OUTSD": "out", "CRLIM": "lim"}).fillna(l["measure_type"].str.lower()))
    pieces.append(l.pivot_table(index=["state", "year"], columns="lab", values="value", aggfunc="sum").reset_index())

    m = pieces[0]
    for p in pieces[1:]:
        m = m.merge(p, on=["state", "year"], how="outer")
    return m

# ---------------- 6. NATIONAL (year-only) SERIES --------------------
ASI_KEEP = ["NBR_FACT", "TOTAL_PERSNS_ENGAGED", "VAL_OF_OUTPUT", "NET_VALUE_ADDED",
            "GRS_FXD_CAP_FORM", "OUT_STND_LNS", "INT_PAID", "INV_CAP"]

def report_years(d):
    d = d.copy()
    yr = d["c1"].astype(str).str.strip()
    d = d[yr.str.match(r"^\d{4}-\d{2}$")].copy()
    d["year"] = d["c1"].str[:4].astype(int) + 1          # '2020-21' -> 2021
    return d.drop_duplicates("year")

def build_national(raw):
    a = raw["asi"].copy()
    a = a[a["inds_pri_char_rn"].isin(ASI_KEEP)]
    a["year"] = a["repyearend"].astype(int)
    a["value"] = num(a["obs_value"]) / np.where(a["unit_measure"] == "INR", CRORE, 1)
    asi = a.pivot_table(index="year", columns="inds_pri_char_rn", values="value", aggfunc="mean")
    asi.columns = ["nat_asi_" + c.lower() for c in asi.columns]

    r = report_years(raw["rrb"])
    rrb = pd.DataFrame({"year": r["year"], "nat_rrb_deposits": num(r["c4"]),
                        "nat_rrb_credit": num(r["c5"])}).set_index("year")
    s = report_years(raw["shg"])
    shg = pd.DataFrame({"year": s["year"], "nat_shg_groups_financed": num(s["c2"]),
                        "nat_shg_bank_loans": num(s["c4"])}).set_index("year")
    return asi.join(rrb, how="outer").join(shg, how="outer").reset_index()

# ---------------- 7. ASSEMBLE + ENGINEER ----------------------------
raw = load_raw()
credit = build_credit(raw["credit"])
macro = build_macro(raw)
nat = build_national(raw)

df = (credit.merge(macro, on=["state", "year"], how="left")
            .merge(nat, on="year", how="left")
            .sort_values(["state", "year"]).reset_index(drop=True))
df.insert(2, "fy", (df["year"] - 1).astype(str) + "-" + (df["year"] % 100).astype(str).str.zfill(2))

sd = lambda a, b: a / b.replace(0, np.nan)
df["deflator"] = sd(df["pc_nsdp_curr"], df["pc_nsdp_cnst"])                 # 2011-12 = 1
df["real_cr_out_total"] = sd(df["cr_out_total"], df["deflator"])
df["real_cr_out_manuf"] = sd(df["cr_out_manuf"], df["deflator"])
df["credit_depth_gsdp"] = sd(df["cr_out_total"], df["gsdp_curr"])
df["credit_depth_nsva"] = sd(df["cr_out_total"], df["nsva_total_curr"])
df["manuf_credit_to_output"] = sd(df["cr_out_manuf"], df["nsdp_manuf_curr"])
df["manuf_share"] = sd(df["nsdp_manuf_curr"], df["nsva_total_curr"])

ln = lambda s: np.log(s.where(s > 0))
for col in ["real_cr_out_manuf", "real_cr_out_total", "nsdp_manuf_cnst",
            "nsdp_manuf_unreg_cnst", "pc_nsdp_cnst"]:
    df["ln_" + col] = ln(df[col])
g = df.groupby("state")
for col in ["ln_real_cr_out_manuf", "ln_nsdp_manuf_cnst", "ln_nsdp_manuf_unreg_cnst"]:
    prev = g[col].shift(1).where(g["year"].diff() == 1)
    df["g_" + col[3:]] = df[col] - prev                                       # log growth
df["l1_ln_real_cr_out_manuf"] = g["ln_real_cr_out_manuf"].shift(1).where(g["year"].diff() == 1)

# H2: initial credit depth (first year with GSDP-based depth, else NSVA-based)
first = lambda col: df.dropna(subset=[col]).sort_values("year").groupby("state")[col].first()
init = first("credit_depth_gsdp").combine_first(first("credit_depth_nsva"))
df["initial_depth"] = df["state"].map(init)
df["initial_depth_tercile"] = df["state"].map(
    pd.qcut(init.rank(method="first"), 3, labels=[1, 2, 3]).astype(int))

# flags
df["boundary_break"] = (
    (df["state"].isin(["Andhra Pradesh"]) & (df["year"] < 2014)) |
    (df["state"].isin(["Jammu & Kashmir"]) & (df["year"] < 2020)) |
    (df["state"] == "Dadra & Nagar Haveli and Daman & Diu")).astype(int)
df["branch_hub"] = df["state"].isin(["Maharashtra", "Delhi"]).astype(int)

df.to_csv(OUT_PATH, index=False)

# ---------------- 8. QUICK REPORT -----------------------------------
print(f"\nSaved {OUT_PATH}  shape={df.shape}  states={df['state'].nunique()}  years={df['year'].min()}-{df['year'].max()}")
cov = df.groupby("year")[["cr_out_manuf", "nsdp_manuf_cnst", "nsdp_manuf_unreg_cnst",
                          "gsdp_curr", "nsva_total_curr", "deflator", "cpi_fy"]].count()
print("\nNon-missing states per year (coverage check):\n", cov.to_string())


read r1222_bank_credit_of_scbs_bank_group_population_group_occupatio.csv         (2244718, 12)
read comp_eco_act_nsdp_fact_cst_rn.csv                                           (34381, 15)
read gross_state_dom_prd_rn.csv                                                  (1788, 13)
read per_cap_net_sdp_rn.csv                                                      (3145, 13)
read cpi_ruc_st_rn.csv                                                           (131594, 13)
read lna_scb_sr_occ_bsr1_a_rn.csv                                                (222, 13)
read ann_surv_inds_rn.csv                                                        (1079, 11)
read r331_important_banking_indicators_regional_rural_banks_outstand.csv         (58, 13)
read r338_self_help_group_bank_linkage_programme.csv                             (45, 11)
  credit: 2,244,711 rows -> 598 state-years, 36 states
  pc_nsdp_cnst: older base not linked (no overlap year) for 1 states
  !! unmapped state codes kept as-is (add them 

## Column dictionary for `master_dataset.csv`
One row = one state × fiscal year (2009-10 to 2025-26). Examples come from your 10% sample; r1222 credit sums were scaled ×10, so they are approximate. Rows marked *illustrative* have no example in the sample because they need complete series.


### Identifiers

| Column | Unit | Meaning | Example |
|---|---|---|---|
| `state` | text | State / UT, harmonised across files (Delhi = DELHI + NCT OF DELHI; DNH and Daman & Diu merged for all years; Odisha = ORI/OD codes) | Karnataka |
| `year` | integer | Fiscal year **ending** in March of this year; credit is outstanding as on 31 March | 2023 |
| `fy` | text | Fiscal-year label | 2022-23 |
| `region` | text | RBI region of the state (from r1222) | Southern Region |
| `n_districts` | count | Districts reporting credit that year (coverage check) | 31 |

### Bank credit, r1222 BSR-1 (Rs crore, nominal)

| Column | Unit | Meaning | Example |
|---|---|---|---|
| `cr_out_total` | Rs crore | Total SCB credit outstanding, all sectors | 10,85,837 |
| `cr_lim_total` | Rs crore | Total sanctioned credit limit | 16,16,093 |
| `cr_acc_total` | accounts | Number of loan accounts, all sectors | 1,53,83,870 |
| `cr_out_agri … cr_out_others` | Rs crore | Outstanding by occupation: I Agriculture, II Industry, III Transport operators, IV Professional & other services, V Personal loans, VI Trade, VII Finance, VIII All others | agri 1,29,412; industry 2,42,624; personal 1,28,303 |
| `cr_out_manuf` | Rs crore | **Manufacturing credit** = Industry sub-sectors 2-16 (food … other industries); excludes mining (1), electricity/gas/water (17), construction (18) | 2,12,057 |
| `cr_lim_manuf` | Rs crore | Sanctioned limit for manufacturing | 4,48,570 |
| `cr_acc_manuf` | accounts | Manufacturing loan accounts | 7,04,790 |
| `cr_out_mining / _utilities / _construction` | Rs crore | Industry sub-sectors 1, 17, 18 kept separately | mining 1,020; utilities 23,456 |
| `avg_ticket_manuf_lakh` | Rs lakh / account | Average manufacturing loan size = outstanding ÷ accounts. Lower = smaller borrowers | 30.1 |
| `avg_ticket_total_lakh` | Rs lakh / account | Average loan size, all sectors | 7.1 |
| `small_ticket_share_manuf` | share 0-1 | Share of manufacturing credit in cells (district × area × bank group × sub-sector) whose average loan ≤ Rs 25 lakh. **Proxy for MSE credit intensity** | 0.04 (sample); typical state 0.1-0.6 |
| `grassroots_share_total` | share 0-1 | Credit from RRBs/SFBs/LABs **or** rural & semi-urban branches ÷ total credit | 0.21 |
| `grassroots_share_manuf` | share 0-1 | Same, for manufacturing credit only | 0.04 |
| `rrb_sfb_share_manuf` | share 0-1 | Manufacturing credit from RRBs + small finance banks + local area banks | 0.003 |
| `rural_semiurban_share_manuf` | share 0-1 | Manufacturing credit booked in rural + semi-urban branches | 0.04 |
| `util_manuf / util_total` | ratio | Utilisation = outstanding ÷ sanctioned limit (≈1 binding, low = idle limits) | 0.47 / 0.67 |

### State income, RBI Handbook (Rs crore unless noted)

| Column | Unit | Meaning | Example |
|---|---|---|---|
| `nsdp_manuf_curr / _cnst` | Rs crore | Net state value added in **manufacturing**, current / constant 2011-12 prices; 2004-05 & 1999-00 bases chain-linked to 2011-12. Main H1 outcome | Karnataka 2025-26 cnst: 2,07,834 |
| `nsdp_manuf_reg_cnst` | Rs crore, 2004-05 prices | Registered manufacturing (old base only, ends 2014-15) | Puducherry 2014-15: 2,458 |
| `nsdp_manuf_unreg_cnst / _curr` | Rs crore | **Unregistered (informal, small-firm) manufacturing**; old base only, ends 2014-15 -> short-panel H1 check | Bihar 2014-15: 4,311 |
| `nsva_total_curr / _cnst` | Rs crore | Total net state value added at basic prices (2011-12 base, 2011-12 onwards) | Odisha 2025-26 curr: 7,51,750 |
| `gsdp_curr / _cnst` | Rs crore | Gross state domestic product, spliced; file ends ≈2017-18 | Karnataka 2017-18 curr: 12,68,881 |
| `pc_nsdp_curr / _cnst` | Rs per person | Per-capita NSDP, spliced to 2011-12 base | J&K 2025-26 curr: 1,68,243 |
| `deflator` | index, 2011-12 = 1 | Implicit state price deflator = pc_nsdp_curr ÷ pc_nsdp_cnst | ~1.6 for 2022-23 (illustrative) |
| `cpi_fy` | index, FY2011-12 = 100 | CPI combined general index, Apr-Mar average (12 months required), 2010 base linked to 2012 base; starts 2011-12 | ~178 for 2022-23 (illustrative) |
| `lna_occ1_out / lna_occ1_lim` | Rs crore | BSR-1 state table, occupation group 1 (= agriculture; matches r1222 agri) - cross-check only | Karnataka 2022-23 out: 1,53,440 |

### Engineered variables

| Column | Unit | Meaning | Example |
|---|---|---|---|
| `real_cr_out_manuf / real_cr_out_total` | Rs crore, 2011-12 prices | Credit ÷ deflator | ~1,32,500 (illustrative) |
| `credit_depth_gsdp` | ratio | Total credit ÷ GSDP (to ≈2017-18) | Karnataka 2017-18: 0.46 |
| `credit_depth_nsva` | ratio | Total credit ÷ total NSVA (2011-12 onwards; use after 2018) | Odisha 2025-26: 0.67 |
| `manuf_credit_to_output` | ratio | Manufacturing credit ÷ manufacturing NSVA (current) | Telangana 2025-26: 0.34 |
| `manuf_share` | share | Manufacturing NSVA ÷ total NSVA (control) | Kerala 2023-24: 0.09 |
| `ln_real_cr_out_manuf, ln_real_cr_out_total, ln_nsdp_manuf_cnst, ln_nsdp_manuf_unreg_cnst, ln_pc_nsdp_cnst` | log | Natural logs for regressions | ln_nsdp_manuf_cnst Karnataka 2025-26: 12.24 |
| `g_real_cr_out_manuf, g_nsdp_manuf_cnst, g_nsdp_manuf_unreg_cnst` | log-diff | Annual growth (only if previous year exists) | g_nsdp_manuf_cnst Goa 2017-18: 0.023 (2.3%) |
| `l1_ln_real_cr_out_manuf` | log | One-year lag of log real manufacturing credit (main FE specification) | - |
| `initial_depth` | ratio | Credit depth in the state's first available year (2009-10 normally) - **H2 interaction** | Karnataka: 0.30 |
| `initial_depth_tercile` | 1/2/3 | Tercile of initial_depth across states (1 = credit-scarce) | 2 |
| `boundary_break` | 0/1 | 1 = territory differs from later years: undivided Andhra Pradesh (<2013-14), J&K incl. Ladakh (<2019-20), merged DNH-DD | 0 |
| `branch_hub` | 0/1 | 1 = Maharashtra, Delhi (credit booked at HQ branches; drop in robustness) | 0 |

### National series (same value for every state in a year; absorbed by year fixed effects)

| Column | Unit | Meaning | Example |
|---|---|---|---|
| `nat_asi_*` | Rs crore / counts | ASI all-India: factories, persons engaged, output, NVA, GFCF, outstanding loans, interest paid, invested capital | nat_asi_val_of_output 2023-24: 1,53,27,166 |
| `nat_rrb_deposits / nat_rrb_credit` | Rs crore | RRB total deposits / credit outstanding (r331) | 2020-21: 5,10,435 / 3,31,277 |
| `nat_shg_groups_financed / nat_shg_bank_loans` | count / Rs crore | SHGs credit-linked and bank loans during the year (r338) | 2008-09 in source file: 16,09,586 / 12,254 (pre-2009-10 years are dropped from the panel) |